In [1]:
# Load and Explore Data
import pandas as pd

# Load data
data = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Display the first few rows
print(data.head())

# Check for missing values
print(data.isnull().sum())

# Get basic info about the data
print(data.info())

# Describe the data
print(data.describe())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [2]:
import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# Load and preprocess data
def load_and_preprocess_data(data_path):
    data = pd.read_csv(data_path)
    
    # Drop unnecessary columns
    data = data.drop(['customerID'], axis=1)
    
    # Convert TotalCharges to numeric, setting errors='coerce' to turn invalid parsing into NaN
    data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
    
    # Fill NaN values in TotalCharges with the mean (or another strategy)
    data['TotalCharges'].fillna(data['TotalCharges'].mean(), inplace=True)

    # Encode the target variable to numerical format
    label_encoder = LabelEncoder()
    data['Churn'] = label_encoder.fit_transform(data['Churn'])

    # One-hot encode categorical features
    categorical_cols = data.select_dtypes(include=['object']).columns
    data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
    
    return data

# Split data into features (X) and target (y)
def split_data(data, target_column):
    X = data.drop(target_column, axis=1)
    y = data[target_column]
    return X, y

if __name__ == "__main__":
    data_path = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

    # Check if the data file exists
    if not os.path.isfile(data_path):
        print(f"Error: Data file not found at {data_path}")
        exit()

    # Load and preprocess the data
    try:
        data = load_and_preprocess_data(data_path)
    except Exception as e:
        print(f"Error preprocessing data: {e}")
        exit()

    # Split the data
    try:
        X, y = split_data(data, "Churn")
    except Exception as e:
        print(f"Error splitting data: {e}")
        exit()

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale numerical features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Start an MLflow run
    with mlflow.start_run():
        # Define hyperparameters
        params = {
            "penalty": "l2",
            "C": 0.1,
            "solver": "liblinear",
            "random_state": 42
        }

        # Log parameters
        mlflow.log_params(params)

        # Train the model
        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        # Make predictions
        y_pred = model.predict(X_test)

        # Evaluate the model
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        # Log the model
        mlflow.sklearn.log_model(model, "model")

        # Print out the model metrics
        print(f"Model accuracy: {accuracy}")
        print(f"Model precision: {precision}")
        print(f"Model recall: {recall}")
        print(f"Model f1-score: {f1}")

/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Model accuracy: 0.8211497515968772
Model precision: 0.6873065015479877
Model recall: 0.5951742627345844
Model f1-score: 0.6379310344827587


In [3]:
import threading
import mlflow
import mlflow.pyfunc
import pandas as pd
from flask import Flask, request, jsonify

mlflow.set_tracking_uri("http://127.0.0.1:5000")
model = mlflow.pyfunc.load_model("runs:/6ae1954bfa2c4a099a680d520f5fd6a5/model")

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data_df = pd.DataFrame(request.get_json())
        return jsonify(model.predict(data_df).tolist())
    except Exception as e:
        return jsonify({"error": str(e)})

threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5002, debug=False, use_reloader=False),
    daemon=True,
).start()
print("Flask running on http://127.0.0.1:5002")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

 * Serving Flask app '__main__'
Flask running on http://127.0.0.1:5002
 * Debug mode: off



 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5002
 * Running on http://172.17.0.2:5002
Press CTRL+C to quit
/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
127.0.0.1 - - [22/Oct/2026 19:46:26] "POST /predict HTTP/1.1" 200 -
/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
127.0.0.1 - - [22/Oct/2026 19:48:31] "POST /predict HTTP/1.1" 200 -


In [4]:
!curl -s -X POST -H "Content-Type: application/json" -d '[{"SeniorCitizen": 0, "tenure": 1, "MonthlyCharges": 29.85, "TotalCharges": 29.85, "Partner_Yes": 1, "Dependents_Yes": 0, "PhoneService_Yes": 0, "MultipleLines_No phone service": 1, "MultipleLines_Yes": 0, "OnlineSecurity_No internet service": 0, "OnlineSecurity_Yes": 0, "OnlineBackup_No internet service": 0, "OnlineBackup_Yes": 1, "DeviceProtection_No internet service": 0, "DeviceProtection_Yes": 0, "TechSupport_No internet service": 0, "TechSupport_Yes": 0, "StreamingTV_No internet service": 0, "StreamingTV_Yes": 0, "StreamingMovies_No internet service": 0, "StreamingMovies_Yes": 0, "PaperlessBilling_Yes": 1, "InternetService_DSL": 0, "InternetService_Fiber optic": 0, "InternetService_No": 0, "Contract_One year": 0, "Contract_Two year": 0, "PaymentMethod_Credit card (automatic)": 0, "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0}]' http://127.0.0.1:5002/predict

[1]


In [5]:
import requests

url = "http://127.0.0.1:5002/predict"
data = [{"SeniorCitizen": 0, "tenure": 1, "MonthlyCharges": 29.85, "TotalCharges": 29.85,
         "Partner_Yes": 1, "Dependents_Yes": 0, "PhoneService_Yes": 0,
         "MultipleLines_No phone service": 1, "MultipleLines_Yes": 0,
         "OnlineSecurity_No internet service": 0, "OnlineSecurity_Yes": 0,
         "OnlineBackup_No internet service": 0, "OnlineBackup_Yes": 1,
         "DeviceProtection_No internet service": 0, "DeviceProtection_Yes": 0,
         "TechSupport_No internet service": 0, "TechSupport_Yes": 0,
         "StreamingTV_No internet service": 0, "StreamingTV_Yes": 0,
         "StreamingMovies_No internet service": 0, "StreamingMovies_Yes": 0,
         "PaperlessBilling_Yes": 1, "InternetService_DSL": 0,
         "InternetService_Fiber optic": 0, "InternetService_No": 0,
         "Contract_One year": 0, "Contract_Two year": 0,
         "PaymentMethod_Credit card (automatic)": 0,
         "PaymentMethod_Electronic check": 1, "PaymentMethod_Mailed check": 0}]

response = requests.post(url, json=data)
print(response.json())   # [1] = predicted to churn, [0] = predicted to stay

[1]
